# EP08 — Multimodal AI: CLIP, BLIP, Fusion Strategies
**Exam Relevance:** 2025 Q8 (11.5 marks) | 2024 Q7 (16 marks)

The highest-marked multimodal question across both years. Cover everything here.

Topics:
1. Modality gap — what it is and why it matters
2. Early fusion vs Late fusion (with advantages)
3. CLIP — contrastive learning to align image and text
4. InfoNCE loss in CLIP
5. BLIP — ITC, ITM, and LM objectives

---

## 1. Modality Gap (2025 Q8.1)

**Definition:** The modality gap is the phenomenon where representations from different modalities (e.g., images vs text) occupy separate, non-overlapping regions of the embedding space, even after being projected into a "shared" space.

**Why it's a problem:**
- Makes cross-modal retrieval harder (similar images and texts are still far apart in embedding space)
- Contrastive learning (like CLIP) is specifically designed to close this gap by pulling matching pairs together and pushing non-matching pairs apart

---

## 2. Early Fusion vs Late Fusion (2025 Q8.1)

| | Early Fusion | Late Fusion |
|---|---|---|
| When | Combine raw/low-level features before any processing | Process each modality separately, combine final representations |
| Input to model | Concatenated or mixed features | Separate encoders → combine at end |
| **Advantage** | **Captures cross-modal interactions at low level** | **Each modality processed by its optimal architecture** |
| Disadvantage | Hard to design a joint architecture | May miss fine-grained cross-modal correlations |
| Example | Concatenate image pixels + text tokens → single transformer | ResNet for images, BERT for text → concatenate CLS tokens |

**Exam tip:** Give exactly ONE advantage of each — that's what they ask.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Demonstrate Early vs Late Fusion architectures
# ============================================================

class EarlyFusionModel(nn.Module):
    """
    Early fusion: concatenate image and text features FIRST,
    then process through shared network.
    """
    def __init__(self, img_dim=64, text_dim=64, hidden=128, n_classes=4):
        super().__init__()
        self.fusion_net = nn.Sequential(
            nn.Linear(img_dim + text_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_classes)
        )
    
    def forward(self, img_feat, text_feat):
        # Merge immediately at input level
        fused = torch.cat([img_feat, text_feat], dim=-1)
        return self.fusion_net(fused)

class LateFusionModel(nn.Module):
    """
    Late fusion: process image and text independently,
    combine only at the final layer.
    """
    def __init__(self, img_dim=64, text_dim=64, hidden=64, n_classes=4):
        super().__init__()
        # Separate encoders for each modality
        self.img_encoder  = nn.Sequential(nn.Linear(img_dim, hidden), nn.ReLU())
        self.text_encoder = nn.Sequential(nn.Linear(text_dim, hidden), nn.ReLU())
        # Shared classifier after fusion
        self.classifier = nn.Linear(hidden * 2, n_classes)
    
    def forward(self, img_feat, text_feat):
        # Each modality processed independently
        img_out  = self.img_encoder(img_feat)
        text_out = self.text_encoder(text_feat)
        # Combine only at the end
        fused = torch.cat([img_out, text_out], dim=-1)
        return self.classifier(fused)

# Count params
early = EarlyFusionModel()
late  = LateFusionModel()

print("Architecture Comparison:")
print(f"  Early Fusion params: {sum(p.numel() for p in early.parameters()):,}")
print(f"  Late Fusion params:  {sum(p.numel() for p in late.parameters()):,}")
print()
print("Early Fusion Architecture:")
print("  [img_feat (64)] ─┐")
print("                    ├─ concat(128) → hidden(128) → output(4)")
print("  [txt_feat (64)] ─┘")
print()
print("Late Fusion Architecture:")
print("  [img_feat (64)] → img_encoder(64) ─┐")
print("                                       ├─ concat(128) → output(4)")
print("  [txt_feat (64)] → txt_encoder(64) ─┘")

Architecture Comparison:
  Early Fusion params: 17,028
  Late Fusion params:  8,836

Early Fusion Architecture:
  [img_feat (64)] ─┐
                    ├─ concat(128) → hidden(128) → output(4)
  [txt_feat (64)] ─┘

Late Fusion Architecture:
  [img_feat (64)] → img_encoder(64) ─┐
                                       ├─ concat(128) → output(4)
  [txt_feat (64)] → txt_encoder(64) ─┘


## 3. CLIP — Contrastive Language-Image Pretraining (2025 Q8.2 + 2024 Q7.1)

### How CLIP aligns image and text representations:

1. **Two separate encoders:** Image encoder (ViT or ResNet) + Text encoder (Transformer)
2. **Project into shared embedding space:** Both produce vectors of the same dimension
3. **Contrastive training with InfoNCE loss:**
   - For a batch of N (image, text) pairs:
   - Compute all N² dot products between image and text embeddings
   - **Positive pairs** (matching image-text): maximise similarity (push together)
   - **Negative pairs** (non-matching): minimise similarity (push apart)
   - The diagonal of the N×N similarity matrix = positive pairs

This is exactly what the CLIP diagram on page 7 of the 2025 exam shows!

In [ ]:
# Implement a minimal CLIP-style contrastive loss (InfoNCE)

def clip_loss(image_embeddings, text_embeddings, temperature=0.07):
    """
    InfoNCE loss used in CLIP.
    
    image_embeddings: (N, D) — one embedding per image
    text_embeddings:  (N, D) — one embedding per text
    Assumes images[i] and texts[i] are matching pairs.
    """
    # L2 normalise
    img_norm  = F.normalize(image_embeddings, dim=-1)
    text_norm = F.normalize(text_embeddings,  dim=-1)
    
    # N x N similarity matrix (all pairs)
    sim_matrix = img_norm @ text_norm.T / temperature
    
    # Labels: diagonal = correct matches (0,0), (1,1), ..., (N-1,N-1)
    labels = torch.arange(sim_matrix.size(0))
    
    # Cross-entropy from image->text direction
    loss_img2text = F.cross_entropy(sim_matrix, labels)
    # Cross-entropy from text->image direction
    loss_text2img = F.cross_entropy(sim_matrix.T, labels)
    
    # Average both directions
    return (loss_img2text + loss_text2img) / 2

# Simulate before and after CLIP training
torch.manual_seed(42)
N, D = 5, 64  # 5 pairs, 64-dim embeddings

# Before training: random, misaligned embeddings
img_before  = torch.randn(N, D)
text_before = torch.randn(N, D)

# After training simulation: aligned (same direction + small noise)
shared = torch.randn(N, D)
img_after  = shared + torch.randn(N, D) * 0.1
text_after = shared + torch.randn(N, D) * 0.1

loss_before = clip_loss(img_before, text_before)
loss_after  = clip_loss(img_after, text_after)

print(f"InfoNCE loss BEFORE alignment: {loss_before.item():.4f}")
print(f"InfoNCE loss AFTER alignment:  {loss_after.item():.4f}")
print()

# Visualise the similarity matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, img_emb, text_emb, title in [
    (axes[0], img_before, text_before, 'Before Training\n(random embeddings)'),
    (axes[1], img_after,  text_after,  'After Training\n(aligned embeddings)'),
]:
    sim = (F.normalize(img_emb, dim=-1) @ F.normalize(text_emb, dim=-1).T).detach().numpy()
    im = ax.imshow(sim, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
    ax.set_xlabel('Text embedding index')
    ax.set_ylabel('Image embedding index')
    ax.set_title(title)
    ax.set_xticks(range(N))
    ax.set_yticks(range(N))
    ax.set_xticklabels([f'Text {i}' for i in range(N)], rotation=45)
    ax.set_yticklabels([f'Image {i}' for i in range(N)])
    plt.colorbar(im, ax=ax, label='Cosine similarity')
    # Highlight diagonal (correct pairs)
    for i in range(N):
        ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, ec='blue', lw=2))

plt.suptitle('CLIP InfoNCE Loss: Diagonal = Positive Pairs (should be high)\nOff-diagonal = Negative Pairs (should be low)', 
             fontsize=11)
plt.tight_layout()
plt.show()

## 4. BLIP — ITC, ITM, LM (2024 Q7.2)

BLIP uses three training objectives simultaneously:

### ITC — Image-Text Contrastive Learning
- Same as CLIP: pull matching (image, text) pairs together, push non-matching apart
- Builds aligned image and text representations in shared space
- Uses a **unimodal encoder** (no cross-attention between modalities)

### ITM — Image-Text Matching
- Binary classification: given an image + text, predict "match" or "no match"
- Uses **cross-attention** between image and text tokens (deeper interaction)
- Learns fine-grained alignment between visual and language concepts

### LM — Language Modelling
- Auto-regressively generate text conditioned on an image
- Uses **causal self-attention** (cannot see future tokens)
- Enables image captioning and visual question answering

**How they collectively contribute:**
- ITC: coarse alignment (which image goes with which text)
- ITM: fine-grained alignment (deeper understanding of content)
- LM: generative capability (can produce text from images)
- Together: BLIP handles both understanding AND generation tasks

In [ ]:
# Visualise BLIP objectives conceptually
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs = [
    {
        'title': 'ITC — Image-Text Contrastive',
        'subtitle': 'Align in shared embedding space',
        'cross_attn': False,
        'input': 'Image + Text separately encoded',
        'output': 'Similarity score (push matching together)',
        'color': '#AED6F1',
        'attention': 'Uni-directional (no cross-modal)',
    },
    {
        'title': 'ITM — Image-Text Matching',
        'subtitle': 'Classify match/no-match',
        'cross_attn': True,
        'input': 'Image + Text with cross-attention',
        'output': 'Binary: match or no-match',
        'color': '#A9DFBF',
        'attention': 'Bi-directional cross-attention',
    },
    {
        'title': 'LM — Language Modelling',
        'subtitle': 'Generate text from image',
        'cross_attn': True,
        'input': 'Image + Text prefix',
        'output': 'Next token prediction (autoregressive)',
        'color': '#F9E79F',
        'attention': 'Causal (masked) cross-attention',
    },
]

for ax, cfg in zip(axes, configs):
    ax.axis('off')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.set_title(cfg['title'], fontsize=11, fontweight='bold')
    
    # Image encoder box
    ax.add_patch(plt.Rectangle((0.5, 5), 3, 1.2, color='#D5DBDB', ec='gray'))
    ax.text(2.0, 5.6, 'Image\nEncoder', ha='center', va='center', fontsize=9)
    
    # Text encoder box
    ax.add_patch(plt.Rectangle((0.5, 2.5), 3, 1.2, color='#D5DBDB', ec='gray'))
    ax.text(2.0, 3.1, 'Text\nEncoder', ha='center', va='center', fontsize=9)
    
    # Cross-attention indicator
    if cfg['cross_attn']:
        ax.annotate('', xy=(3.5, 5.6), xytext=(3.5, 3.7), 
                   arrowprops=dict(arrowstyle='<->', color='red', lw=2))
        ax.text(4.2, 4.65, 'Cross\nAttn', ha='center', fontsize=8, color='red')
    
    # Output box
    ax.add_patch(plt.Rectangle((5.5, 3.5), 4, 1.5, color=cfg['color'], ec='gray'))
    ax.text(7.5, 4.35, cfg['output'], ha='center', va='center', fontsize=8, wrap=True)
    
    # Arrow to output
    ax.annotate('', xy=(5.5, 4.25), xytext=(3.5, 4.25),
               arrowprops=dict(arrowstyle='->', color='black'))
    
    # Attention type label
    ax.text(5.0, 1.5, cfg['attention'], ha='center', fontsize=8, 
            style='italic', color='gray',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('BLIP Pre-training Objectives\n(Three tasks trained simultaneously)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("BLIP summary:")
print("  ITC: coarse alignment (which image↔text match globally)")
print("  ITM: fine-grained matching (deep cross-modal understanding)")
print("  LM:  generation (caption images, answer questions)")

## 5. MCQ Answers (2025 Q5.5 + 2024)

**2025 Q5.5:** *How do multimodal models typically represent inputs from different modalities?*  
**Answer: A** — By projecting them into a shared embedding space to enable cross-modal interaction

---

## Exam Quick-Reference Summary

**Modality gap:** Representations from different modalities (image vs text) are in different regions of embedding space — contrastive learning closes this gap.

**Early fusion:** Combine modalities early → advantage: captures low-level cross-modal interactions  
**Late fusion:** Combine modalities late → advantage: each modality processed by its optimal encoder

**CLIP alignment (how it works — 3 key steps):**
1. Encode image and text separately into shared D-dim vectors
2. Compute N×N similarity matrix for all (image, text) pairs in batch
3. InfoNCE loss: maximise diagonal (matching pairs), minimise off-diagonal (non-matching)

**BLIP three objectives:**
| Objective | Type | What it learns |
|---|---|---|
| ITC | Contrastive | Coarse image-text alignment |
| ITM | Binary classification | Fine-grained matching/mismatch detection |
| LM | Generation | Produce text from image (captioning) |